In [1]:
import random
import numpy as np
import pandas as pd
import torch
from transformers import BertTokenizer, BertModel

random.seed(2026)
np.random.seed(2026)
torch.manual_seed(2026)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [2]:
train_df = pd.read_csv('/kaggle/input/alchemy-aicc-round-4/train.csv')
test_df = pd.read_csv('/kaggle/input/alchemy-aicc-round-4/test.csv')
cand_df = pd.read_csv('/kaggle/input/alchemy-aicc-round-4/candidates.csv')
candidate_labels = sorted(cand_df['result'].unique().tolist())

print(f'Train: {len(train_df)} rules')
print(f'Test: {len(test_df)} pairs')
print(f'Candidates: {len(candidate_labels)} unique results')

Train: 150 rules
Test: 70 pairs
Candidates: 70 unique results


In [3]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased').to(device).eval()

def get_embedding(text):
    enc = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=32)
    with torch.no_grad():
        out = model(**{k: v.to(device) for k, v in enc.items()})
    return out.last_hidden_state[:, 0, :].cpu().numpy()

cand_embs = np.vstack([get_embedding(c) for c in candidate_labels])

pair_embs = np.vstack([get_embedding(f"{row['item1']} {row['item2']}") for _, row in test_df.iterrows()])

pair_norm = pair_embs / np.linalg.norm(pair_embs, axis=1, keepdims=True)
cand_norm = cand_embs / np.linalg.norm(cand_embs, axis=1, keepdims=True)
score_matrix = pair_norm @ cand_norm.T

print(f'Score matrix shape: {score_matrix.shape}')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Score matrix shape: (70, 70)


In [4]:
col_indices = []
used = set()

for row in score_matrix:
    for c in np.argsort(-row): 
        if c not in used:
            col_indices.append(c)
            used.add(c)
            break

In [5]:
predictions = [candidate_labels[c] for c in col_indices]

submission = pd.DataFrame({
    'Id': test_df['Id'],
    'result': predictions
})
submission.to_csv('submission.csv', index=False)
print(f'Submission shape: {submission.shape}')
print(submission.head())

Submission shape: (70, 2)
   Id       result
0   0    satellite
1   1          mud
2   2   salt water
3   3        wheat
4   4  necromancer
